# OCR v7.3.2 FINAL — Layout estrutural (VS Code + Jupyter)

PaddleOCR-VL **doc_parser** completo: análise de layout local + reconhecimento VLM no `llama-server` (RX 580) → SQLite/FTS5 + JSON canônico por página com blocos, bounding boxes e risco.

## Preparo no VS Code

**1. Instale a extensão "Jupyter" da Microsoft no VS Code.**

**2. Crie o kernel do venv (uma vez):**

source ~/.venvs/paddle-layout/bin/activate
pip install -U "paddlepaddle==3.2.1" -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
pip install ipykernel
python -m ipykernel install --user --name paddle-layout --display-name "Python (paddle-layout)"

**3. No VS Code, abra este notebook e clique em "Select Kernel" → "Python (paddle-layout)".**

**4. Terminal separado — servidor VLM na GPU (deixar rodando):**

# O modelo foi baixado para ia-sh/models (o ~/models e read-only neste ambiente).
# Use o script start_vlm_server.sh ou o comando abaixo (CPU; use -ngl 99 se tiver GPU):
llama-server -m /home/igor/ia-sh/models/PaddleOCR-VL-1.6/PaddleOCR-VL-1.6-GGUF.gguf \
  --mmproj /home/igor/ia-sh/models/PaddleOCR-VL-1.6/PaddleOCR-VL-1.6-GGUF-mmproj.gguf \
  --host 127.0.0.1 --port 8080 -c 16384 -ngl 0 -np 4 --temp 0

In [1]:
# --- Entrada e saída -----------------------------------------------------------
PDF_PATH = "documento_teste.pdf"     # PDF de entrada
OUT_DIR = "v73_teste"                # saída: SQLite, FTS, JSONs canônicos
IMPORT_ONLY = False                  # True: só reimporta JSONs já gerados

# --- Servidor VLM (RX 580) -----------------------------------------------------
URL = "http://127.0.0.1:8081/v1"     # llama-server PaddleOCR-VL (porta 8081)
MODEL = "/home/igor/ia-sh/models/PaddleOCR-VL-1.6/PaddleOCR-VL-1.6-GGUF.gguf"  # ID exato registrado pelo llama-server (caminho do .gguf)
VLM_CONCURRENCY = 4                  # deve coincidir com o -np do llama-server

# --- Layout local ----------------------------------------------------------------
DEVICE = "cpu"                       # layout local; "gpu" só após validar na Polaris

## 0. Ambiente — comandos para iniciar a execução

Tudo abaixo roda **fora** do notebook (no terminal do VS Code, separado).

**1) Criar o ambiente — primeira vez apenas:**
```bash
python3 -m venv ~/.venvs/paddle-layout
source ~/.venvs/paddle-layout/bin/activate
pip install -U pip
pip install "paddlepaddle==3.2.1" -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
pip install "paddleocr[doc-parser]>=3.6.0"
pip install ipykernel
python -m ipykernel install --user --name paddle-layout --display-name "Python (paddle-layout)"
deactivate
```

**2) Registrar o kernel no VS Code — primeira vez apenas:**
```bash
source ~/.venvs/paddle-layout/bin/activate
python -m ipykernel install --user --name paddle-layout --display-name "Python (paddle-layout)"
deactivate
```
> No notebook: **Select Kernel → "Python (paddle-layout)"**.

**3) Subir o servidor VLM — a cada execução (terminal separado, deixar rodando):**
```bash
bash /home/igor/ia-sh/start_vlm_server.sh
```
> CPU (`-ngl 0`), porta 8080, `-np 4`. Se um dia houver GPU NVIDIA, troque `-ngl 99` no script.

**Conferir a saúde do servidor (opcional):**
```bash
curl -s http://127.0.0.1:8080/health && echo
curl -s http://127.0.0.1:8080/v1/models | python3 -m json.tool
```

**Só depois** volte ao notebook e rode a **célula 3 (Pré-voo)**, que valida PDF, kernel, servidor e `MODEL`.


In [6]:
import hashlib, json, re, sqlite3, time, logging, tempfile, shutil
from pathlib import Path

log = logging.getLogger("ocr_v73")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

SCHEMA_VERSION = 3
COMPLEX = {"table", "formula", "chart", "seal", "formula_number"}
NO_TEXT = {"image", "figure", "chart", "seal"}

def db_open(path: Path) -> sqlite3.Connection:
    con = sqlite3.connect(path)
    con.execute("PRAGMA journal_mode=WAL")
    con.execute("PRAGMA synchronous=NORMAL")       # ==> melhoria: WAL + NORMAL = escrita mais rápida
    con.execute("PRAGMA foreign_keys=ON")
    con.executescript("""
    CREATE TABLE IF NOT EXISTS meta(key TEXT PRIMARY KEY, value TEXT NOT NULL);
    CREATE TABLE IF NOT EXISTS documents(
      id TEXT PRIMARY KEY, path TEXT NOT NULL, pages INTEGER NOT NULL,
      created_at TEXT DEFAULT CURRENT_TIMESTAMP);
    CREATE TABLE IF NOT EXISTS pages(
      document_id TEXT NOT NULL, page INTEGER NOT NULL, markdown TEXT NOT NULL,
      risk INTEGER NOT NULL, PRIMARY KEY(document_id,page),
      FOREIGN KEY(document_id) REFERENCES documents(id) ON DELETE CASCADE);
    CREATE TABLE IF NOT EXISTS blocks(
      document_id TEXT NOT NULL, page INTEGER NOT NULL, ord INTEGER NOT NULL,
      label TEXT NOT NULL, score REAL NOT NULL,
      x1 REAL NOT NULL, y1 REAL NOT NULL, x2 REAL NOT NULL, y2 REAL NOT NULL,
      content TEXT NOT NULL, risk INTEGER NOT NULL, reasons_json TEXT NOT NULL,
      PRIMARY KEY(document_id,page,ord),
      FOREIGN KEY(document_id) REFERENCES documents(id) ON DELETE CASCADE);
    CREATE INDEX IF NOT EXISTS idx_blocks_risk ON blocks(risk DESC);   # ==> melhoria: consulta top-risk rápida
    CREATE VIRTUAL TABLE IF NOT EXISTS blocks_fts USING fts5(
      document_id UNINDEXED, page UNINDEXED, ord UNINDEXED, content,
      tokenize='unicode61 remove_diacritics 2');
    """)
    return con

def check_schema(con):
    row = con.execute("SELECT value FROM meta WHERE key='schema_version'").fetchone()
    if row and int(row[0]) != SCHEMA_VERSION:
        raise RuntimeError(f"schema v{row[0]} incompatível com v{SCHEMA_VERSION}; use outro OUT_DIR")
    con.execute("INSERT OR REPLACE INTO meta VALUES('schema_version',?)", (str(SCHEMA_VERSION),))
    con.commit()

def sha256(pdf: Path) -> str:
    h = hashlib.sha256()
    with open(pdf, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def atomic(path: Path, text: str) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)

def _walk_coords(item, flat):
    """Achata recursivamente listas/tuplas/dicts de coordenadas."""
    if isinstance(item, dict):                       # ==> fix: suporta dict de pontos
        for v in item.values():
            _walk_coords(v, flat)
    elif isinstance(item, (list, tuple)):
        for v in item:
            _walk_coords(v, flat)
    elif isinstance(item, (int, float)) and not isinstance(item, bool):
        flat.append(float(item))

def rect(coordinate):
    """Aceita bbox, quadrilátero, polígono ou dict; retorna envelope retangular.
    Tolerante a None e a folhas não numéricas (nunca lança)."""
    flat = []
    _walk_coords(coordinate or [], flat)
    if len(flat) < 4 or len(flat) % 2:
        return 0.0, 0.0, 0.0, 0.0
    xs, ys = flat[::2], flat[1::2]
    return min(xs), min(ys), max(xs), max(ys)

def reading_order(boxes):
    """Fallback determinístico: largos separam faixas; estreitos por coluna/topo.
    ==> fix: banda processa faixa vertical inteira entre anchors; nenhum narrow é perdido.
    O markdown oficial do doc_parser continua sendo a fonte de verdade textual."""
    if not boxes:
        return []
    xmin = min(b["bbox"][0] for b in boxes)
    xmax = max(b["bbox"][2] for b in boxes)
    width = max(1.0, xmax - xmin)
    wide_ids = {b["source_index"] for b in boxes
                if b["bbox"][2] - b["bbox"][0] >= 0.70 * width}
    wide = [b for b in boxes if b["source_index"] in wide_ids]
    narrow = [b for b in boxes if b["source_index"] not in wide_ids]
    anchors = sorted(wide, key=lambda b: b["bbox"][1])
    result = []
    def column_key(b):
        return round((b["bbox"][0] + b["bbox"][2]) / 2.0 / (0.05 * width))
    def append_band(items):
        cols = sorted({column_key(b) for b in items})
        for col in cols:
            result.extend(sorted(
                [b for b in items if column_key(b) == col],
                key=lambda b: (b["bbox"][1], b["source_index"])))
    # Faixas verticais:
    #   [0..anchors[0].y1)         → acima do 1º anchor
    #   [anchors[i-1].y3..anchor.y1) → entre dois anchors
    #   narrows DENTRO da extensão vertical de um anchor → anexados logo após o anchor
    #   [último.y3..inf)           → abaixo do último anchor
    # Nenhum narrow é perdido (bug corrigido); ordem é mais fiel ao layout.
    remaining = list(narrow)
    def take_band(lo, hi, in_anchor=False):
        nonlocal remaining
        items, keep = [], []
        for b in remaining:
            if lo <= b["bbox"][1] < hi:
                items.append(b)
            else:
                keep.append(b)
        remaining = keep
        if in_anchor:  # ordena por y1; sem agrupar em colunas (já estamos dentro de um anchor)
            result.extend(sorted(items, key=lambda b: (b["bbox"][1], b["source_index"])))
        else:
            append_band(items)
    prev_bottom = -float("inf")
    for anchor in anchors:
        take_band(prev_bottom, anchor["bbox"][1])   # narrows acima do anchor
        result.append(anchor)
        # narrows cuja base y1 está DENTRO da extensão vertical do anchor
        take_band(anchor["bbox"][1], anchor["bbox"][3], in_anchor=True)
        prev_bottom = anchor["bbox"][3]
    take_band(prev_bottom, float("inf"))           # narrows abaixo do último anchor
    append_band(remaining)                          # segurança: sobras (nunca ocorre)
    return result

def content_of(box: dict) -> str:
    for key in ("block_content", "content", "text", "markdown"):
        v = box.get(key)
        if isinstance(v, str):
            return v
        if isinstance(v, dict):                       # algumas versões anexam dicts
            return v.get("markdown_text") or v.get("text") or ""
    return ""

def markdown_of(res: dict) -> str:
    md = res.get("markdown")
    if isinstance(md, dict):                          # paddleocr 3.7 pode aninhar
        return md.get("markdown_text") or md.get("text") or ""
    if isinstance(md, str):
        return md
    return res.get("markdown_text") or ""

def normalize_page(raw: dict, page: int) -> dict:
    res = raw.get("res", raw)
    layout = res.get("layout_det_res", {}) or {}
    boxes = []
    for index, box in enumerate(layout.get("boxes", []) or []):
        boxes.append({
            "source_index": index,
            "label": str(box.get("label") or "unknown"),
            "score": float(box.get("score") or 0),
            "bbox": rect(box.get("coordinate") or box.get("bbox") or []),
            "content": content_of(box),
        })
    ordered = reading_order(boxes)
    markdown = (markdown_of(res)
                or "\n\n".join(b["content"] for b in ordered if b["content"]))
    return {"page": page, "markdown": markdown, "blocks": ordered}

def block_risk(block: dict):
    score, reasons = 0, []
    if block["score"] < 0.50:
        score += 40; reasons.append("layout_score_baixo")
    if block["label"] in COMPLEX:
        score += 20; reasons.append("elemento_complexo")
    content = str(block["content"] or "")          # ==> fix: blindagem contra None
    if not content and block["label"] not in NO_TEXT:
        score += 35; reasons.append("bloco_sem_conteudo")
    if re.search(r"[?]{2,}", content):
        score += 30; reasons.append("caracteres_suspeitos")
    return min(score, 100), reasons

def ingest(con, did: str, data: dict) -> None:
    page = data["page"]
    with con:
        con.execute("DELETE FROM blocks WHERE document_id=? AND page=?", (did, page))
        con.execute("DELETE FROM blocks_fts WHERE document_id=? AND page=?", (did, page))
        risks = []
        rows, fts_rows = [], []
        for order, block in enumerate(data["blocks"]):
            risk, reasons = block_risk(block)
            risks.append(risk)
            x1, y1, x2, y2 = block["bbox"]
            rows.append((did, page, order, block["label"], block["score"], x1, y1, x2, y2,
                         block["content"], risk, json.dumps(reasons, ensure_ascii=False)))
            if block["content"]:
                fts_rows.append((did, page, order, block["content"]))
        # ==> melhoria: batch executemany (menos round-trips)
        if rows:
            con.executemany(
                "INSERT INTO blocks VALUES(?,?,?,?,?,?,?,?,?,?,?,?)", rows)
        if fts_rows:
            con.executemany(
                "INSERT INTO blocks_fts VALUES(?,?,?,?)", fts_rows)
        con.execute("INSERT OR REPLACE INTO pages VALUES(?,?,?,?)",
                    (did, page, data["markdown"], max(risks, default=0)))

def run_parser(pdf: Path, out: Path) -> Path:
    """API Python oficial do PaddleOCR-VL — layout local + VLM via llama-server.
    Retorna o subdiretório com os JSONs desta execução."""
    from paddleocr import PaddleOCRVL  # import adiado: IMPORT_ONLY não exige paddleocr
    pipeline = PaddleOCRVL(
        pipeline_version="v1.6",
        vl_rec_backend="llama-cpp-server",
        vl_rec_server_url=URL,
        vl_rec_api_model_name=MODEL,
        vl_rec_max_concurrency=VLM_CONCURRENCY,
        use_layout_detection=True,
        format_block_content=True,
        layout_shape_mode="rect",
        use_queues=True,
        device=DEVICE,
    )
    # ==> melhoria: subdiretório com timestamp evita misturar JSONs de execuções antigas
    stamp = time.strftime("%Y%m%d_%H%M%S")
    json_dir = out / "parser_json" / stamp
    json_dir.mkdir(parents=True, exist_ok=True)
    for res in pipeline.predict(input=str(pdf)):
        res.save_to_json(save_path=str(json_dir))
        res.save_to_markdown(save_path=str(json_dir))
    return json_dir

## 3. Pré-voo
Confere PDF, `paddle`/`paddleocr` no kernel, saúde do servidor, existência do `MODEL` e gravabilidade da saída.

In [3]:
import requests

pdf = Path(PDF_PATH).expanduser().resolve()
if not pdf.is_file():
    raise FileNotFoundError(f"PDF não encontrado: {pdf}")

if not URL.rstrip("/").endswith("/v1"):
    log.warning("URL deve terminar em /v1 (ex.: http://127.0.0.1:8080/v1)")

if not IMPORT_ONLY:
    try:
        import paddle          # noqa: F401  (engine de layout local)
        import paddleocr       # noqa: F401
        paddle_version = paddle.__version__
    except ImportError as e:
        raise RuntimeError(
            f"{e}. Instale o engine PaddlePaddle no venv:\n"
            "  source ~/.venvs/paddle-layout/bin/activate\n"
            "  pip install -U 'paddlepaddle==3.2.1' "
            "-i https://www.paddlepaddle.org.cn/packages/stable/cpu/")

    base = URL.split("/v1")[0].rstrip("/")
    try:
        resp = requests.get(base + "/health", timeout=10)
        ok = resp.ok or resp.status_code == 404  # 404 = build sem /health
    except requests.RequestException:
        ok = False
    if not ok:
        raise RuntimeError(f"llama-server fora do ar em {base} — suba com "
                           f"PaddleOCR-VL-1.6 + mmproj antes de rodar")

    # ==> fix: trata falha/JSON inválido em /models
    try:
        resp = requests.get(URL + "/models", timeout=10)
        resp.raise_for_status()
        ids = [m.get("id") for m in resp.json().get("data", [])]
    except (requests.RequestException, ValueError) as e:
        raise RuntimeError(f"falha ao consultar /models: {e}")
    if MODEL not in ids:
        raise RuntimeError(f"MODEL='{MODEL}' não existe no servidor. "
                           f"IDs disponíveis: {ids}. Ajuste MODEL na célula 1.")
    log.info("PaddlePaddle %s | servidor OK | MODEL '%s' existe", paddle_version, MODEL)

out_check = Path(OUT_DIR).resolve()
out_check.mkdir(parents=True, exist_ok=True)
probe = out_check / ".probe"; probe.write_text("ok"); probe.unlink()
log.info("PDF: %s (%.1f MB)", pdf, pdf.stat().st_size / 1e6)
log.info("saída gravável: %s", out_check)

FileNotFoundError: PDF não encontrado: /home/igor/ia-sh/documento_teste.pdf

In [4]:
pdf = Path(PDF_PATH).expanduser().resolve()
if not pdf.is_file():
    raise FileNotFoundError(f"PDF não encontrado: {pdf}")
out = Path(OUT_DIR).resolve()
out.mkdir(parents=True, exist_ok=True)
t0 = time.time()

# ==> melhoria: importa apenas JSONs desta execução (evita misturar execuções antigas)
if not IMPORT_ONLY:
    log.info("rodando doc_parser (primeira vez baixa modelos de layout)...")
    parser_dir = run_parser(pdf, out)
    candidatos = sorted(parser_dir.glob("*.json"))
else:
    log.info("modo import-only: reimportando JSONs canônicos .v73.json")
    # No import-only, volta a varrer o diretório (JSONs canônicos são a fonte)
    candidatos = sorted(out.rglob("*.v73.json"))

if not candidatos:
    raise RuntimeError(f"nenhum JSON do doc_parser em {out}")

did = sha256(pdf)
con = db_open(out / "ocr_v73.sqlite3")
check_schema(con)
con.execute("INSERT OR IGNORE INTO documents(id,path,pages) VALUES(?,?,0)",
            (did, str(pdf)))
con.commit()

imported, ignorados, vistos = [], [], set()
try:
    for fallback, file in enumerate(candidatos, 1):
        try:
            raw = json.loads(file.read_text(encoding="utf-8"))
            if not isinstance(raw, dict) or not ("res" in raw or "layout_det_res" in raw):
                if IMPORT_ONLY and raw.get("page") is not None:
                    # já é canônico (page/markdown/blocks)
                    data = raw
                else:
                    ignorados.append((file.name, "JSON sem estrutura de parser (res/layout_det_res)"))
                    continue
            else:
                index = raw.get("res", raw).get("page_index")
                page = int(index) + 1 if index is not None else fallback
                data = normalize_page(raw, page)
            page = int(data["page"])
            if page < 1:                             # ==> fix: valida página >= 1
                ignorados.append((file.name, f"page_index inválida ({page})"))
                continue
            if page in vistos:
                log.warning("página %d duplicada (%s); última ocorrência prevalece", page, file.name)
            vistos.add(page)
            atomic(out / f"page-{page:05d}.v73.json",
                   json.dumps(data, ensure_ascii=False, indent=2))
            ingest(con, did, data)
        except (json.JSONDecodeError, UnicodeDecodeError) as e:
            ignorados.append((file.name, str(e)[:80]))
            continue
        except Exception as e:
            ignorados.append((file.name, f"{type(e).__name__}: {e}"[:120]))
            continue
        imported.append(page)
finally:
    con.close()   # ==> fix: garante fechamento mesmo com exceção

if not imported:
    raise RuntimeError(f"nenhum JSON válido importado. Ignorados: {ignorados}")

con = db_open(out / "ocr_v73.sqlite3")
con.execute("UPDATE documents SET pages=? WHERE id=?", (len(set(imported)), did))
con.commit()
con.close()
log.info("importadas %d páginas em %.1fs", len(set(imported)), time.time() - t0)
if ignorados:
    log.warning("ignorados %d arquivos não-parser: %s", len(ignorados), ignorados[:5])
log.info("SQLite/FTS: %s", out / "ocr_v73.sqlite3")

FileNotFoundError: PDF não encontrado: /home/igor/ia-sh/documento_teste.pdf

In [5]:
con = db_open(Path(OUT_DIR).resolve() / "ocr_v73.sqlite3")

n_pages = con.execute("SELECT count(*) FROM pages").fetchone()[0]
if n_pages == 0:                                     # ==> melhoria: aviso claro se célula 5 não rodou
    print("[aviso] nenhuma página importada — rode a célula 5 antes.")

print("blocos por tipo:")
for label, n in con.execute(
        "SELECT label, count(*) FROM blocks GROUP BY label ORDER BY count(*) DESC"):
    print(f"  {label:20s} {n}")

print("\nblocos mais arriscados (top 10):")
for page, order, label, risk, reasons in con.execute(
        "SELECT page, ord, label, risk, reasons_json FROM blocks "
        "ORDER BY risk DESC, page, ord LIMIT 10"):
    try:                                             # ==> melhoria: reasons corrompido não derruba
        reasons = json.loads(reasons)
    except (json.JSONDecodeError, TypeError):
        reasons = ["<reasons inválido>"]
    print(f"  pág {page} bloco {order}: {label} risco={risk} {reasons}")

print("\nmarkdown da página 1 (prévia):")
row = con.execute("SELECT markdown FROM pages ORDER BY page LIMIT 1").fetchone()
print((row[0] if row else "")[:500])

print("\nbusca FTS — troque o termo:")
termo = "licitação"   # aspas duplas para frases ex.: '"termo exato"'
# ==> melhoria: escapa aspas duplas e usa MATCH protegido
try:
    q = termo.replace('"', '""')
    for page, order, snip in con.execute(
            "SELECT page, ord, snippet(blocks_fts,3,'[',']','…',16) "
            "FROM blocks_fts WHERE blocks_fts MATCH ? LIMIT 5", (q,)):
        print(f"  pág {page} bloco {order}: {snip}")
except sqlite3.OperationalError as e:
    print(f"  [busca FTS ignorada: {e}]")
con.close()

OperationalError: unable to open database file